# 2.1 Drone SDK Encapsulation

1. **Controlling a drone like a human would**

    Using a large language model to control a drone is similar to how a pilot operates one. Making the operation simpler is key to applying LLMs effectively.

2. **The raw API is too complex**

    Currently, even a simple takeoff in AirSim requires calling multiple API functions, designed for system stability and usability. But for an LLM or a pilot, a single "take off" command is all that's needed.


----

Based on research and practice, it is generally necessary to wrap the drone SDK (or similar unmanned system SDK) with a secondary layer to make it easier for LLMs to call. Common design rules include:


1. **Semantic interface wrapping**
Encapsulate low-level coordinate conversions (e.g., AirSim's negative Z-axis) inside the interface, exposing natural semantics externally (e.g., `fly_to([x, y, positive_altitude])`), similar to the Z-value handling logic in `fly_to()`.

2. **Atomic functionality**
Each method corresponds to an independent operation unit (e.g., takeoff/land wrapped separately), avoiding compound operations so the LLM can call them as needed (similar to the separation of `set_yaw()` and `fly_to()`).

3. **Standardized parameters**
Use basic data types (list/dict) uniformly for input/output. For example, positions always return as [x, y, z] lists, reducing complexity for the LLM (see the `get_drone_position()` return value design).

4. **Silent error handling**
Built-in retry mechanisms and default return values, such as the while loop in `get_position()` to avoid empty values, preventing call chain interruptions.

5. **Unified physical units**
Convert units internally (e.g., radians to degrees), always using intuitive units externally (e.g., `get_yaw()` returns degrees).

6. **Context persistence**
Maintain client connection state (initialized in `__init__`), saving session context through class attributes to ensure continuity across multiple command calls.

7. **Semantic alias mapping**
Establish an object dictionary (objects_dict) to convert natural language to engine objects, improving prompt compatibility.

8. **Async-to-sync conversion**
Convert async operations to sync via `.join()` (e.g., `takeoffAsync().join()`), avoiding the complexity of handling async callbacks in the LLM.

9. **Sensor abstraction**
Pre-process raw sensor data (e.g., LiDAR point clouds), returning directly usable metrics (`get_distance()` returns the minimum distance value).

10. **Safety boundary settings**
Maintain API control in critical operations (e.g., reset) through mechanisms like `enableApiControl` to prevent loss of control.

Suggested additional rules: add precondition checks before method calls (e.g., verify armed status before takeoff), standardize return structures (include a success flag uniformly), and add a debug mode toggle to further improve LLM call reliability.

Some typical encapsulation patterns are as follows:

In [ ]:
def takeoff(self):
    """
    takeoff the drone
    """
    self.client.takeoffAsync().join()

def land(self):
    """
    land the drone
    """
    self.client.landAsync().join()

    
def get_yaw(self):
    """
    get the yaw angle of the drone
    :return: yaw_degree, the yaw angle of the drone in degree
    """
    orientation_quat = self.client.simGetVehiclePose().orientation
    yaw = airsim.to_eularian_angles(orientation_quat)[2] # get the yaw angle
    yaw_degree = math.degrees(yaw)
    return yaw_degree # return the yaw angle in degree

See the airsim_wrapper.py code for details.

When using dedicated frameworks later, there will be more specific requirements for function wrapping, such as:

In [ ]:
@tool
def suggest_menu(occasion: str) -> str:
    """
    Suggests a menu based on the occasion.
    Args:
        occasion: The type of occasion for the party.
    """
    if occasion == "casual":
        return "Pizza, snacks, and drinks."
    elif occasion == "formal":
        return "3-course dinner with wine and dessert."
    elif occasion == "superhero":
        return "Buffet with high-energy and healthy food."
    else:
        return "Custom menu for the butler."

@tool
def get_position(object_name:str)->[float, float, float]:
    """
    get the position of a specific object
    :param object_name: the name of the object
    :return: position, the position of the object
    """
    query_string = objects_dict[object_name] + ".*"
    object_names_ue = []
    while len(object_names_ue) == 0:
        object_names_ue = self.client.simListSceneObjects(query_string)
    pose = self.client.simGetObjectPose(object_names_ue[0])
    return [pose.position.x_val, pose.position.y_val, pose.position.z_val]


## SDK Code Test - Launch the AirSim Simulator

In [ ]:
# Launch the simulator
import airsim_wrapper
aw = airsim_wrapper.AirSimWrapper()
aw.takeoff()
aw.fly_to([20, 0, -10])
print("done")

In [ ]:
aw.land()